# Quantitative Finance: Explainable Models and GPU Machine Learning

## Research question and target

> **Can recent price momentum, volatility, volume behavior, and sector predict whether an S&P 500 stock has a positive return in the next trading session?**

**Target (y):** `next_day_up` (1 if the next-session return is positive, otherwise 0).

**Inputs (X):** one-day return, five-day and 20-day momentum, 20-day realized volatility, five-day log-volume change, and sector. These are known at the prediction-date close. Future returns and price fields are excluded from the predictors.

The notebook compares a **majority-class baseline**, **Logistic Regression** (interpretable coefficients), a **shallow Decision Tree** (visible decision rules), a **Random Forest** (ensemble of trees), and **GPU-accelerated XGBoost** (nonlinear boosted trees). Splits are chronological by date, and boundary rows whose next-day labels cross into another split are purged. The latest test period stays untouched.

## 1. Load the prepared feature panel

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from google.colab import drive

drive.mount('/content/drive')
DATA_DIR=Path('/content/drive/MyDrive/sp500_kaggle_processed')
FEATURE_PATH=DATA_DIR/'sp500_next_day_features.parquet'
assert FEATURE_PATH.exists(),'Run the S&P 500 EDA notebook and create its feature panel first.'
data=pd.read_parquet(FEATURE_PATH)
data['date']=pd.to_datetime(data['date'])
data['target_date']=pd.to_datetime(data['target_date'])
print('Rows:',len(data),'| symbols:',data['symbol'].nunique(),'| dates:',data['date'].min().date(),'to',data['date'].max().date())
display(data.head())

### Why inspect the panel before training

Confirm that the feature panel includes multiple symbols and a long date range. Because the Kaggle dataset refreshes, the newest label date can change after each download.

## 2. Define X and y

In [ ]:
TARGET='next_day_up'
NUMERIC_FEATURES=['return_1d','momentum_5d','momentum_20d','volatility_20d','volume_change_5d']
CATEGORICAL_FEATURES=['sector']
FEATURES=NUMERIC_FEATURES+CATEGORICAL_FEATURES
required=['date','target_date',TARGET,*FEATURES]
missing=sorted(set(required)-set(data.columns))
if missing: raise KeyError(f'Missing fields: {missing}')
model_data=data[required].replace([np.inf,-np.inf],np.nan).dropna(subset=[TARGET,'date']).sort_values('date').reset_index(drop=True)
model_data[TARGET]=model_data[TARGET].astype(int)
print('Target class share:');display(model_data[TARGET].value_counts(normalize=True).rename('share').to_frame())
display(model_data[FEATURES].describe(include='all').T)

### How X and y are defined

`X` contains five lagged price or volume measures and sector; `y` is the next-session direction. The future return and its date are kept only for label construction and split-boundary protection. Missing predictor values are handled inside each model pipeline.

## 3. Chronological train, validation, and test split

All symbols on the same trading date stay together. The earliest 70% of distinct feature dates form training, the next 15% validation, and the latest 15% test. Since each label uses the following session, purge training rows whose target date reaches validation and validation rows whose target date reaches test. This prevents information from crossing split boundaries.

In [ ]:
dates=np.array(sorted(model_data['date'].dt.normalize().unique()))
if len(dates)<30: raise ValueError(f'Only {len(dates)} unique dates; not enough for this split.')
cut_train=max(1,int(len(dates)*.70))
cut_valid=max(cut_train+1,int(len(dates)*.85))
cut_valid=min(cut_valid,len(dates)-1)
train_dates,valid_dates,test_dates=dates[:cut_train],dates[cut_train:cut_valid],dates[cut_valid:]
train=model_data[model_data['date'].dt.normalize().isin(train_dates)].copy()
valid=model_data[model_data['date'].dt.normalize().isin(valid_dates)].copy()
test=model_data[model_data['date'].dt.normalize().isin(test_dates)].copy()
valid_start=pd.Timestamp(valid_dates[0])
test_start=pd.Timestamp(test_dates[0])
train=train[train['target_date']<valid_start].copy()
valid=valid[valid['target_date']<test_start].copy()
if min(len(train),len(valid),len(test))==0: raise ValueError('A chronological partition is empty after purging split-boundary labels.')
split_info=[]
for name,part in [('train',train),('validation',valid),('test (held out)',test)]:
    split_info.append({'split':name,'rows':len(part),'first_feature_date':part['date'].min().date(),
                       'last_feature_date':part['date'].max().date(),'up_rate':part[TARGET].mean()})
display(pd.DataFrame(split_info))
assert train['date'].max()<valid['date'].min()
assert valid['date'].max()<test['date'].min()
assert train['target_date'].max()<valid['date'].min()
assert valid['target_date'].max()<test['date'].min()

### Why the split is chronological and purged

A random split could let future market regimes influence training. Grouping by date keeps all symbols from one session together. Purging removes a training or validation row when its next-session label would fall inside the next partition, reducing look-ahead across boundaries. Read the printed dates and row counts to confirm the intended order.

## 4. Establish the majority-class baseline

In [ ]:
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,confusion_matrix,classification_report

def classification_metrics(y_true,y_pred):
    return {'accuracy':accuracy_score(y_true,y_pred),
            'precision':precision_score(y_true,y_pred,zero_division=0),
            'recall':recall_score(y_true,y_pred,zero_division=0),
            'f1':f1_score(y_true,y_pred,zero_division=0)}
majority_class=int(train[TARGET].mode().iloc[0])
baseline_pred=np.full(len(valid),majority_class)
baseline_metrics=classification_metrics(valid[TARGET],baseline_pred)
print('Training majority class:',majority_class)
print('Validation baseline:',baseline_metrics)

### What the majority baseline tells you

The baseline always predicts the more common training class. Its accuracy can look respectable when up and down days are imbalanced, while its recall for the other class may be zero. Treat it as a minimum benchmark, not a useful trading strategy.

## 5. Fit explainable models

Logistic Regression estimates how each standardized feature relates to the odds of an up day. A shallow Decision Tree uses a few explicit if/then rules. Random Forest averages many trees; it can capture nonlinear patterns but is less transparent than one small tree. Preprocessing is learned from training data only. The Random Forest uses a fixed, reproducible sample of at most 500,000 training rows to limit CPU runtime; validation remains the same for every model.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

numeric_pipe=Pipeline([('imputer',SimpleImputer(strategy='median')),('scale',StandardScaler())])
sector_pipe=Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),
                      ('onehot',OneHotEncoder(handle_unknown='ignore'))])
def make_preprocessor():
    return ColumnTransformer([('numeric',numeric_pipe,NUMERIC_FEATURES),
                              ('sector',sector_pipe,CATEGORICAL_FEATURES)])

logistic_model=Pipeline([('preprocess',make_preprocessor()),
                         ('classifier',LogisticRegression(max_iter=500))])
tree_model=Pipeline([('preprocess',make_preprocessor()),
                     ('classifier',DecisionTreeClassifier(max_depth=3,min_samples_leaf=100,random_state=42))])
# Limit the CPU Random Forest to a reproducible sample from training only.
rf_train=train.sample(n=min(500_000,len(train)),random_state=42)
forest_model=Pipeline([('preprocess',make_preprocessor()),
                       ('classifier',RandomForestClassifier(n_estimators=150,max_depth=12,
                           min_samples_leaf=50,n_jobs=-1,random_state=42))])

logistic_model.fit(train[FEATURES],train[TARGET])
tree_model.fit(train[FEATURES],train[TARGET])
forest_model.fit(rf_train[FEATURES],rf_train[TARGET])
logistic_pred=logistic_model.predict(valid[FEATURES])
tree_pred=tree_model.predict(valid[FEATURES])
forest_pred=forest_model.predict(valid[FEATURES])
logistic_metrics=classification_metrics(valid[TARGET],logistic_pred)
tree_metrics=classification_metrics(valid[TARGET],tree_pred)
forest_metrics=classification_metrics(valid[TARGET],forest_pred)
print('Logistic Regression:',logistic_metrics)
print('Shallow Decision Tree:',tree_metrics)
print('Random Forest (training rows:',len(rf_train),'):',forest_metrics)

### Why compare these three explainable models

Logistic Regression is a simple linear reference whose coefficients can be inspected. The depth-limited Decision Tree gives a small set of explicit threshold rules. Random Forest averages many trees to capture nonlinear patterns; a fixed sample of at most 500,000 training rows limits CPU time. All three are scored on the same validation dates.

## 6. Fit a GPU-accelerated model: XGBoost

XGBoost builds trees in sequence, with each new tree improving errors from earlier trees. We use its histogram tree method on CUDA. In Colab, first choose **Runtime → Change runtime type → GPU**. This is a strong GPU-friendly tabular candidate, not a guarantee of best performance.

In [ ]:
%pip -q install xgboost
import subprocess
import xgboost as xgb

gpu_check=subprocess.run(['nvidia-smi'],capture_output=True,text=True)
if gpu_check.returncode!=0:
    raise RuntimeError('No NVIDIA GPU detected. In Colab choose Runtime > Change runtime type > GPU, reconnect, and rerun.')
print(gpu_check.stdout.splitlines()[0])
print('XGBoost version:',xgb.__version__)

xgb_model=Pipeline([('preprocess',make_preprocessor()),
    ('classifier',xgb.XGBClassifier(
        n_estimators=300,max_depth=6,learning_rate=0.05,
        subsample=0.8,colsample_bytree=0.8,reg_lambda=1.0,
        objective='binary:logistic',eval_metric='logloss',importance_type='gain',
        tree_method='hist',device='cuda',random_state=42,n_jobs=-1
    ))])
xgb_model.fit(train[FEATURES],train[TARGET])
xgb_pred=xgb_model.predict(valid[FEATURES])
xgb_metrics=classification_metrics(valid[TARGET],xgb_pred)
print('GPU XGBoost validation:',xgb_metrics)

### What successful GPU training means

The cell checks for an NVIDIA runtime before fitting. `tree_method='hist'` builds histogram-based splits, and `device='cuda'` requests CUDA execution. This makes XGBoost the GPU model in the comparison; it does not guarantee better validation performance. If Colab reports no GPU, change the runtime type to GPU and reconnect.

## 7. Compare models on the same validation period

Accuracy measures total correct classifications; precision measures how often predicted up days were actually up; recall measures the share of actual up days found; F1 balances precision and recall. Compare each model with the training-only majority baseline.

In [ ]:
comparison=pd.DataFrame([
    {'model':'Majority-class baseline',**baseline_metrics},
    {'model':'Logistic Regression',**logistic_metrics},
    {'model':'Shallow Decision Tree',**tree_metrics},
    {'model':'Random Forest',**forest_metrics},
    {'model':'GPU XGBoost',**xgb_metrics},
]).set_index('model')
display(comparison.round(4))
print('Use validation results for model selection; do not use the held-out test set yet.')
print('Held-out test date range:',test['date'].min().date(),'to',test['date'].max().date(),'- not evaluated.')
comparison.to_csv('/content/drive/MyDrive/sp500_validation_metrics.csv')
print('Saved validation comparison to Drive.')

### Validation charts: score comparison and error types

The first chart places all models on the same validation metrics. The second chart shows the logistic model's correct and incorrect label counts. These plots are diagnostic summaries, not financial return curves.

In [ ]:
# Compare model validation metrics in a common panel.
ax=comparison[['accuracy','precision','recall','f1']].plot(kind='bar',figsize=(12,6),ylim=(0,1),rot=20)
ax.set_title('Validation metrics by model')
ax.set_ylabel('Score')
ax.grid(axis='y',alpha=.25)
plt.tight_layout();plt.show()

# A confusion matrix shows which class errors remain for Logistic Regression.
from sklearn.metrics import ConfusionMatrixDisplay
fig,ax=plt.subplots(figsize=(5,4))
ConfusionMatrixDisplay.from_predictions(valid[TARGET],logistic_pred,
    display_labels=['down/flat','up'],cmap='Blues',ax=ax,colorbar=False)
ax.set_title('Logistic Regression: validation confusion matrix')
plt.tight_layout();plt.show()

### Reading the validation plots

The grouped bars make it easy to see whether one model raises F1 at the expense of precision or recall. The confusion matrix has actual labels on one axis and predictions on the other; the diagonal cells are correct classifications, while off-diagonal cells are mistakes. Compare the logistic model's pattern with the baseline table before describing a gain. The plot reports validation counts, not probabilities or profits.

### How to choose from the validation table

Compare accuracy, precision, recall, and F1 with the baseline. Use F1 when balancing positive-class precision and recall matters, and inspect the class balance before interpreting accuracy. Prefer the simplest model whose validation results meet your project goal. Do not use the held-out test dates to choose among models.

## 8. Explain and defend the model choices

- **Majority baseline:** always predicts the most common training class; it sets a simple performance bar.
- **Logistic Regression:** a linear classifier. Coefficient signs show how each standardized feature relates to the log-odds of a positive next-session return, with other features held fixed.
- **Shallow Decision Tree:** a few learned threshold rules. Capping depth at three makes the decision path easy to show.
- **Random Forest:** averages many trees, which can reduce the instability of one tree and capture nonlinear patterns. It is less transparent than the shallow tree; feature importance is a summary, not a causal explanation.
- **GPU XGBoost:** boosted trees model nonlinear feature combinations. `tree_method='hist'` and `device='cuda'` request CUDA training. It is compared with the other models rather than assumed to be best.
- **Fair evaluation:** the target and validation dates are the same for every model; only training data fit preprocessing and model parameters. The final test dates remain unused.

In [ ]:
# Logistic Regression coefficients: positive values increase estimated log-odds of an up day.
feature_names=logistic_model.named_steps['preprocess'].get_feature_names_out()
coef=pd.Series(logistic_model.named_steps['classifier'].coef_[0],index=feature_names)
print('Largest absolute Logistic Regression coefficients:')
display(coef.reindex(coef.abs().sort_values(ascending=False).index).head(12).to_frame('coefficient'))

# Shallow tree: display the learned if/then thresholds.
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree
fig,ax=plt.subplots(figsize=(18,8))
plot_tree(tree_model.named_steps['classifier'],
          feature_names=tree_model.named_steps['preprocess'].get_feature_names_out(),
          class_names=['down/flat','up'],filled=True,rounded=True,fontsize=8,ax=ax)
ax.set_title('Shallow Decision Tree (maximum depth 3)')
fig.tight_layout();plt.show()

# XGBoost gain importance summarizes split utility; it does not show causation.
xgb_names=xgb_model.named_steps['preprocess'].get_feature_names_out()
xgb_importance=pd.Series(xgb_model.named_steps['classifier'].feature_importances_,index=xgb_names).sort_values(ascending=False).head(12)
display(xgb_importance.rename('importance').to_frame())
xgb_importance.sort_values().plot(kind='barh',figsize=(8,5),title='Top XGBoost feature importances')
plt.xlabel('Importance');plt.tight_layout();plt.show()

### How to explain the coefficient, tree, and importance outputs

A positive Logistic Regression coefficient raises the estimated log-odds of an up day when other inputs are held fixed; numeric inputs were standardized. The shallow tree visualization shows explicit split thresholds and class proportions. XGBoost gain importance summarizes which inputs helped its splits. None of these summaries shows causation or guarantees future performance.

## Limitations and next steps

- Kaggle prices update over time; record the dataset version and download date for reproducibility.
- The historical series follows currently represented S&P 500 constituents, creating survivorship bias.
- Next-day direction is noisy; class balance and market-wide dependence matter.
- Logistic Regression and the small Decision Tree are easiest to explain. Random Forest and XGBoost are more flexible but less transparent.
- XGBoost uses GPU histogram training (`tree_method='hist', device='cuda'`). Speed depends on the Colab GPU and data size; the notebook checks for an available NVIDIA GPU.
- After selecting a model from validation, evaluate once on the held-out test dates. Then consider walk-forward validation, historical index membership, and transaction costs.

# Detailed modeling notes and defense guide

## Why next-session direction is the target

The Kaggle file is a daily price dataset, so its directly observable outcome is a future price return. The model asks a narrow classification question: at the close of date *t*, will the same symbol's next recorded session have a positive return? This makes the target concrete and aligns it with the market data. It does not claim to predict the size of the return, identify an overvalued company, or produce a complete trading strategy.

For each symbol, the EDA notebook calculates `next_day_return` by shifting the daily return series one row toward the past, then labels it as positive/non-positive. The ML notebook excludes the numerical future return and keeps the binary direction label. The future date is retained only to enforce the chronological boundary purge.

## Why the majority classifier is a baseline

A baseline is a transparent reference point. The majority-class baseline finds which target label is most frequent in the training period and predicts that label for every validation row. It uses no technical features. If a complex model cannot improve on this simple rule, the evidence for value from the features is weak.

The baseline can score well on accuracy when one class is more common. That is why accuracy is shown beside precision, recall, and F1. A baseline that always predicts an up day can have zero ability to recognize down days. Read the confusion matrix and class proportions together.

## Logistic Regression in plain language

Logistic Regression estimates the probability of class 1 through a linear score passed through the logistic function:

`p(up) = 1 / (1 + exp(-z))`, where `z = intercept + sum(coefficient_j * feature_j)`.

The coefficients are learned from training data. Numeric inputs are standardized so one feature's units do not dominate simply because it is measured on a larger numerical scale. Sector is converted to indicator columns by one-hot encoding. A positive coefficient increases the model's log-odds for an up label, conditional on the other inputs. It is an association inside this fitted model, not evidence that the feature causes a return.

Logistic Regression is a useful explainable benchmark because its form is simple, its probability can be inspected, and coefficient direction can be discussed. It may miss interactions and nonlinear thresholds, such as momentum mattering only when volatility is high.

## Shallow Decision Tree

A classification tree asks a sequence of questions such as “is 20-day momentum below this threshold?” and then routes observations to a leaf. Each leaf predicts a class based on its training observations. The notebook caps depth at three and requires a minimum leaf size. These limits reduce the size of the diagram and discourage a tree with tiny leaves that memorize individual observations.

A tree is easy to show and narrate, but a small tree may underfit. If the chart shows only a few levels, that is intentional: this is the interpretable reference model, not necessarily the most accurate candidate. Thresholds are learned from this training sample; they are not universal financial rules.

## Random Forest

Random Forest trains many decision trees using randomized samples and feature subsets, then aggregates their votes. Averaging can make predictions less sensitive to one tree's particular splits and capture nonlinear patterns. It is more flexible than the shallow tree, but the combined forest is harder to narrate as a single set of rules.

The notebook uses a reproducible sample of up to 500,000 training rows for the CPU Random Forest. The validation set remains unchanged, so the reported score still evaluates the same period. Explain this runtime decision in a presentation: it keeps a large daily panel manageable in a Colab CPU while retaining a fixed validation comparison. A future extension can test a larger sample or GPU forest implementation if compute and package compatibility allow.

Random Forest feature importance is not causal and can favor variables with many possible split points. Correlated features can divide or mask importance. Use it as a summary of the fitted forest, then compare with the simpler coefficients and tree rules.

## GPU XGBoost

XGBoost builds trees sequentially. Each new tree is fit to improve the errors of the accumulated ensemble. Its tree structure handles nonlinear thresholds and combinations of features, which often suits structured tabular data. The notebook sets `tree_method='hist'` and `device='cuda'` to request GPU histogram training.

The model is a GPU-friendly candidate, not an automatic winner. A GPU runtime must be enabled in Colab, and XGBoost must be compatible with the runtime's CUDA driver. The code checks for an NVIDIA device before fitting. If that check fails, choose a GPU runtime and reconnect. If XGBoost then reports a CUDA compatibility error, follow the XGBoost installation guidance for the runtime's CUDA version rather than silently running a CPU model and calling it GPU training.

The chosen values—300 estimators, depth 6, learning rate 0.05, row/column subsampling, and L2 regularization—are a starting point. This week's goal is a baseline comparison, not extensive tuning. Record the settings so another student can reproduce the run.

## Metrics: formulas and how to read them

Let TP be predicted-up and actually-up rows; TN predicted-down and actually-down; FP predicted-up but actually-down; FN predicted-down but actually-up.

- **Accuracy** = `(TP + TN) / (TP + TN + FP + FN)`. Overall fraction correct; can hide minority-class failure.
- **Precision** = `TP / (TP + FP)`. Among predicted up days, the fraction that were up.
- **Recall** = `TP / (TP + FN)`. Among actual up days, the fraction identified as up.
- **F1** = `2 * precision * recall / (precision + recall)`. Harmonic balance between precision and recall.

Precision and recall in this notebook treat “up” as the positive class. If the project instead prioritizes identifying negative days, the positive label and metric interpretation would need to change. Do not interpret classification metrics as profitability: they ignore return magnitude, transaction costs, slippage, risk exposure, and portfolio construction.

## Time split and boundary protection

Financial examples have temporal dependence. A random train/test split can train on later market regimes and validate on earlier ones. This notebook sorts distinct feature dates and assigns the earliest 70% to training, the next 15% to validation, and the latest 15% to test. All symbols for a date remain together.

Because each feature-date row is labeled with the next session, a row at the end of training may have a target on the first validation date. Such a row is purged from training. The same rule applies from validation into test. This prevents the target observation itself from crossing the partition boundary. The chronological test dates stay untouched until model selection is complete.

The data still has dependence across adjacent dates and symbols. This split is a first benchmark, not a complete statistical guarantee. A stronger follow-up is rolling walk-forward evaluation across multiple non-overlapping time windows, possibly with an additional embargo around split dates.

## How to explain the validation comparison

Start with the target class share and baseline score. Then compare each model's validation metrics. State which model had the best F1 and whether it also maintained useful precision and recall. If accuracy is slightly higher but recall falls, describe that tradeoff instead of claiming an unconditional improvement. If no model beats the baseline, report that honestly and discuss feature noise, target difficulty, or data limitations.

A defensible summary has four parts:

1. **Question:** the model predicts next-session direction from information available at today's close.
2. **Design:** use lagged price, volume, and sector features; split dates chronologically; purge cross-boundary labels.
3. **Benchmark:** compare a no-feature majority rule with interpretable linear/rule models, a Random Forest, and a GPU tree-boosting model.
4. **Finding:** report the actual validation metrics after running the notebook. Do not insert expected scores or claim a model is better before seeing the output.

## Result notes to complete after execution

- Kaggle data version or download date: `________________________`
- Effective feature-date range: `________________________`
- Train / validation / test row counts: `________________________`
- Up-day proportion in training and validation: `________________________`
- Best validation model by the metric selected for the project: `________________________`
- How it compared with the majority baseline: `________________________`
- Important limitation visible in the result: `________________________`
- Next test-set evaluation date: `after model selection is complete`

These prompts are intentionally blank. They should be filled with the output from your own run, not guessed in advance.

# Oral defense and model review appendix

## One-minute project explanation

“My model asks whether information available at today's close can classify the next recorded session's return direction for an S&P 500 stock. The inputs are lagged one-day return, five- and 20-day momentum, 20-day realized volatility, five-day log-volume change, and sector. The target is binary next-day direction. I use a chronological split so later market observations cannot train a model evaluated on earlier dates, and I purge labels that cross split boundaries. I compare a majority-class baseline with Logistic Regression, a small Decision Tree, a sampled Random Forest, and GPU XGBoost. I use validation metrics to choose a candidate and keep the final date block untouched until that choice is made.”

This explanation accurately states the target, inputs, split, baseline, candidate models, metrics, and test discipline. It does not claim that the model is profitable or that one model won before the notebook is run.

## Questions an instructor may ask

### Why classify direction instead of predicting the exact return?

A one-session return is continuous and noisy. This project defines a simple classification target that can be compared against a majority baseline. The trade-off is that classification discards return size. A future regression extension could predict the next-session return or a multi-session risk measure, but would need its own target definition and regression metrics.

### Why use a majority baseline?

It shows how well one can score without using predictors. If up days are more common than down days, always predicting up can achieve decent accuracy. A candidate should be compared on precision, recall, and F1 as well as accuracy.

### Why is Logistic Regression included if it may be too simple?

It gives an interpretable linear reference. The coefficient signs and magnitudes make it easier to explain how the standardized inputs relate to predicted up probability. Its simplicity is useful even if a nonlinear model later performs better.

### Why include both a Decision Tree and a Random Forest?

A shallow tree is the explainable rule model: it can be drawn and described path by path. A Random Forest combines many trees and can capture more patterns, but the ensemble is harder to explain. They answer different needs: clarity versus flexibility.

### Why use XGBoost on a GPU?

The project has a large panel and XGBoost supports CUDA histogram tree training. It is a practical GPU-friendly tabular candidate. The GPU setting is explicitly requested in the estimator, and the notebook checks for NVIDIA hardware. GPU execution is a computation choice, not evidence that the model is better.

### What does feature importance mean?

It summarizes how the fitted tree ensemble used variables to improve its splits. It does not show that a variable causes returns, and correlated predictors can share importance. Use it as a description of the fitted model, not as an economic law.

### Why is the test set left alone?

Choosing features, thresholds, or models after looking at test metrics turns the test set into another validation set. Keeping it untouched preserves a final, later-period check after the model decision is complete.

### What does a negative R² mean in the previous regression version?

R² can be negative when a model's predictions perform worse than predicting the evaluation mean. In this classification version, R² is not used; the notebook reports classification metrics instead.

## Hyperparameter notes

- `max_depth` limits how many split decisions an individual tree can make. Larger depth can fit more complex interactions but can overfit.
- `min_samples_leaf` prevents terminal leaves from being based on very few training rows.
- `n_estimators` sets the number of Random Forest trees or boosting rounds, depending on model family.
- `learning_rate` controls how much each new XGBoost tree changes the ensemble; smaller rates generally need more trees.
- `subsample` samples training rows for each boosted tree, while `colsample_bytree` samples input columns. These settings can reduce over-reliance on a single sample or feature.
- `reg_lambda` adds L2 regularization to the boosted trees.
- The values in this notebook are starting values for a benchmark, not optimized settings. Report them and avoid tuning against test metrics.

## When to prefer each candidate

| Situation | Candidate to explain | Why |
|---|---|---|
| Need a plain benchmark | Majority class | No feature processing and easy to reproduce. |
| Need a coefficient story | Logistic Regression | Linear score and inspectable coefficients. |
| Need visible rules | Shallow Decision Tree | Small number of learned thresholds. |
| Need nonlinear patterns on CPU | Random Forest | Aggregated trees; feature summaries available. |
| Need a GPU tabular candidate | XGBoost | CUDA histogram training and nonlinear boosted trees. |

There is no universally best algorithm independent of data, target, hardware, metric, and time split. The validation table provides evidence for this project's current setup. It does not establish that the winning model will continue to win in a new market regime.

## Model report template

After running the notebook, write a concise paragraph such as: “The majority baseline achieved ___ accuracy and ___ F1 on the validation dates. The Logistic Regression, shallow tree, Random Forest, and GPU XGBoost achieved ___, ___, ___, and ___ F1, respectively. The selected validation candidate was ___ because ___. Its main trade-off was ___. The test period remains untouched. These results describe next-session direction classification and do not measure trading profitability.”

Fill each blank from `comparison`; if results are mixed, explain the metric trade-off. Never write that GPU XGBoost improved performance solely because it used the GPU. GPU acceleration affects training computation; validation metrics describe predictive performance.

## Responsible interpretation of a weak result

A result near the majority baseline is a plausible outcome for one-day direction. Markets contain noise, and the selected lagged features may not carry a strong short-horizon signal. Weak performance can still be informative: it shows that these features and this target do not beat a simple reference in the chosen validation period. Avoid repeatedly changing the target or trying many feature combinations after seeing validation until the result appears favorable; document each planned experiment and retain the test period for final evaluation.

## Final reproducibility checklist

- [ ] Kaggle dataset version or refresh date is recorded.
- [ ] Target and feature columns are written down.
- [ ] Features use no information after the prediction close.
- [ ] Rows are split by date, not randomly by row.
- [ ] Cross-boundary next-session labels are purged.
- [ ] Baseline and candidate models share the same validation period.
- [ ] Random Forest sampling is reproducible and comes only from training data.
- [ ] XGBoost reports CUDA training on a GPU runtime.
- [ ] Metrics are saved and summarized without inventing results.
- [ ] The held-out test period is not used to choose the model.

## How to present charts and code in your lab notebook

Keep the validation metric table close to the code that computes it. Then show the grouped metric chart and confusion matrix, followed by a written interpretation. The tree visualization should follow the fitted shallow tree; the coefficient table should follow Logistic Regression; XGBoost importance should follow GPU training. This ordering lets a reader connect each model's output to the exact code that created it.

When you run the notebook in Colab, save it after all chart cells finish. The notebook will then contain rendered figures as well as executable code and the explanatory notes. For a reproducible submission, keep the Kaggle refresh date, training/validation/test date ranges, and model settings alongside those outputs. Validation charts are part of model selection; do not add test-period charts until the final model has been selected.

## Model comparison integrity

All validation metrics in the comparison table refer to the same date block and the same binary target. The CPU Random Forest uses a reproducible subset of training rows, while its validation rows are unchanged. If a later experiment changes the feature set, target horizon, or dates, rerun the baseline under that same experiment and label the comparison clearly. Keep the model-selection notes with the saved charts; a feature-importance bar chart by itself is not a complete result.